In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load Iris dataset
data = load_iris()
X = data.data
y = data.target

# Normalize input features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# One-hot encode the target
encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y.reshape(-1, 1))

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Define sizes
input_size = X_train.shape[1]      # 4
hidden_size = 10
output_size = y_train.shape[1]     # 3

# Initialize weights and biases
np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

# Activation functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # Stability fix
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Loss function: Cross-entropy
def cross_entropy_loss(y_pred, y_true):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-9), axis=1))  # Add epsilon for stability

# Training loop
learning_rate = 0.01
epochs = 500

for epoch in range(epochs):
    # Forward pass
    Z1 = np.dot(X_train, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)  # Final prediction

    # Loss
    loss = cross_entropy_loss(A2, y_train)

    # Backpropagation
    dZ2 = A2 - y_train                             # Derivative of loss w.r.t. Z2 (3 terms in chain rule collapsed here)
    dW2 = np.dot(A1.T, dZ2) / X_train.shape[0]
    db2 = np.mean(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X_train.T, dZ1) / X_train.shape[0]
    db1 = np.mean(dZ1, axis=0, keepdims=True)

    # Update weights
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    # Print loss every 50 epochs
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# Test accuracy
Z1_test = np.dot(X_test, W1) + b1
A1_test = relu(Z1_test)
Z2_test = np.dot(A1_test, W2) + b2
A2_test = softmax(Z2_test)

y_pred = np.argmax(A2_test, axis=1)
y_true = np.argmax(y_test, axis=1)
accuracy = np.mean(y_pred == y_true)
print(f"\nTest Accuracy: {accuracy:.4f}")


Epoch 0, Loss: 1.2293
Epoch 50, Loss: 0.6185
Epoch 100, Loss: 0.4524
Epoch 150, Loss: 0.3913
Epoch 200, Loss: 0.3542
Epoch 250, Loss: 0.3259
Epoch 300, Loss: 0.3031
Epoch 350, Loss: 0.2841
Epoch 400, Loss: 0.2684
Epoch 450, Loss: 0.2552

Test Accuracy: 0.9333


In [22]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # for stability
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def ann_numpy_classifier(X_train, y_train, X_test, y_test, epochs=1000, hidden_size=8, learning_rate=0.01):
    np.random.seed(42)

    input_size = X_train.shape[1]
    output_size = y_train.shape[1]

    # Weight and bias initialization
    W1 = np.random.randn(input_size, hidden_size)
    b1 = np.zeros((1, hidden_size))

    W2 = np.random.randn(hidden_size, output_size)
    b2 = np.zeros((1, output_size))

    # Training loop
    for epoch in range(epochs):
        # Forward pass
        Z1 = np.dot(X_train, W1) + b1
        A1 = sigmoid(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        # Loss (cross-entropy)
        loss = -np.mean(np.sum(y_train * np.log(A2 + 1e-8), axis=1))

        # Backward pass
        dZ2 = A2 - y_train
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * sigmoid_derivative(Z1)
        dW1 = np.dot(X_train.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update weights
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

        if epoch % 100 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    # ---- Prediction ----
    def predict(X):
        A1 = sigmoid(np.dot(X, W1) + b1)
        A2 = softmax(np.dot(A1, W2) + b2)
        return np.argmax(A2, axis=1)

    y_pred = predict(X_test)
    y_true = np.argmax(y_test, axis=1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return y_pred, (W1, b1, W2, b2)
